In [1]:
import pandas as pd

In [2]:
import re
import ast
import pandas as pd
import numpy as np

def _parse_kinase_coord(x):
    """
    Accepts things like:
      - [[13, 271]]
      - "[[13, 271]]"
      - "13,271"  (fallback)
    Returns list of (start,end) tuples (1-indexed, inclusive).
    """
    if isinstance(x, list):
        coord = x
    elif isinstance(x, str):
        s = x.strip()
        try:
            coord = ast.literal_eval(s)
        except Exception:
            nums = list(map(int, re.findall(r"-?\d+", s)))
            if len(nums) >= 2:
                coord = [[nums[0], nums[1]]]
            else:
                coord = []
    else:
        coord = []

    pairs = []
    for item in coord:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            pairs.append((int(item[0]), int(item[1])))
    return pairs

def _extract_15mer_centered(seq: str, p_ind_1: int, p_aa: str, pad_char: str = ""):
    """
    15-aa window centered at p_ind_1 (1-indexed). Inclusive window [p-7, p+7].
    Pads with pad_char if near termini.
    Returns (window, ok_match) where ok_match indicates seq[p_ind_1-1]==p_aa (if possible).
    """
    if not isinstance(seq, str) or len(seq) == 0 or pd.isna(p_ind_1):
        return (np.nan, False)

    p = int(p_ind_1)
    if p < 1:
        return (np.nan, False)

    ok = True
    if isinstance(p_aa, str) and len(p_aa) == 1 and p <= len(seq):
        ok = (seq[p-1] == p_aa)

    left = p - 7
    right = p + 7

    chars = []
    for i in range(left, right + 1):
        if 1 <= i <= len(seq):
            chars.append(seq[i-1])
        else:
            chars.append(pad_char)
    return ("".join(chars), ok)

def _extract_domain(seq: str, coords, join_char: str = ""):
    """
    Extract kinase domain based on coords = list of (start,end) 1-indexed inclusive.
    If multiple segments, concatenates them in order (default join_char="").
    """
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    if not coords:
        return np.nan

    parts = []
    for (start, end) in coords:
        if start < 1 or end < 1:
            continue
        if end < start:
            start, end = end, start
        parts.append(seq[start-1:end])
    if not parts:
        return np.nan
    return join_char.join(parts)

def _collapse_conflicting_labels(
    df_train: pd.DataFrame,
    key_cols=("kinase_seq", "substrate_seq"),
    label_col="label",
):
    """
    If the same (kinase_seq, substrate_seq) appears with both label 0 and 1,
    keep only the label==1 rows for that pair.

    - Does NOT merge/average scores; it just filters.
    - If a pair appears only with label 0 -> keep (all) 0 rows
      (typically there should be only one row, but we don't assume).
    - If a pair appears only with label 1 -> keep (all) 1 rows.
    """
    # Normalize label to numeric {0,1}
    df = df_train.copy()
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df.dropna(subset=[label_col])
    df[label_col] = df[label_col].astype(int)

    # Identify keys that have any positive
    has_pos = (
        df.groupby(list(key_cols))[label_col]
          .max()
          .rename("_has_pos")
          .reset_index()
    )
    df = df.merge(has_pos, on=list(key_cols), how="left")

    # If a key has a positive, drop the negatives for that key
    df = df.loc[~((df["_has_pos"] == 1) & (df[label_col] == 0))].copy()
    df = df.drop(columns=["_has_pos"], errors="ignore").reset_index(drop=True)
    return df

def make_training_sequences(
    df_raw: pd.DataFrame,
    substrate_seq_col: str = "substrate_seq",
    kinase_seq_col: str = "kinase_seq",
    p_ind_col: str = "p_ind",
    p_aa_col: str = "p_aa",
    kinase_coord_col: str = "kinase_coord",
    label_col: str = "label",
    pad_char: str = "",
    drop_mismatch: bool = False,
    collapse_conflicting_labels: bool = True,
    kinase_domain_ready = False
):
    """
    Returns:
      df_out with new columns:
        - substrate_seq_15aa
        - kinase_domain_seq
        - p_aa_match (bool)
      plus a minimal df_train with columns:
        - kinase_seq (domain)
        - substrate_seq (15aa)
        - label

    Extra behavior:
      - If collapse_conflicting_labels=True, then for any (kinase_seq, substrate_seq)
        pair that has both label 0 and 1, only keep the label==1 rows.
    """
    df = df_raw.copy()

    # parse kinase coords
    df["_kinase_pairs"] = df[kinase_coord_col].apply(_parse_kinase_coord)

    # kinase domain seq
    if not kinase_domain_ready:
        df["kinase_domain_seq"] = [
            _extract_domain(seq, pairs)
            for seq, pairs in zip(df[kinase_seq_col], df["_kinase_pairs"])
        ]
    else:
        df["kinase_domain_seq"] = df['kinase_seq']
    # substrate 15mer
    out = [
        _extract_15mer_centered(seq, pind, paa, pad_char=pad_char)
        for seq, pind, paa in zip(df[substrate_seq_col], df[p_ind_col], df[p_aa_col])
    ]
    df["substrate_seq_15aa"] = [x[0] for x in out]
    df["p_aa_match"] = [x[1] for x in out]

    # optionally drop rows where p_aa doesn't match the sequence at p_ind
    if drop_mismatch:
        df = df[df["p_aa_match"] == True].copy()

    # minimal training df for your pipeline
    df_train = df[["kinase_domain_seq", "substrate_seq_15aa", label_col]].copy()
    df_train = df_train.rename(
        columns={
            "kinase_domain_seq": "kinase_seq",
            "substrate_seq_15aa": "substrate_seq",
            label_col: "label",
        }
    )

    # drop NAs created by bad coords / bad indices
    df_train = df_train.dropna(subset=["kinase_seq", "substrate_seq", "label"]).reset_index(drop=True)

    # NEW: remove conflicts (keep only label==1 if both 0 and 1 exist for same pair)
    if collapse_conflicting_labels:
        df_train = _collapse_conflicting_labels(
            df_train,
            key_cols=("kinase_seq", "substrate_seq"),
            label_col="label",
        )

    # cleanup helper
    df = df.drop(columns=["_kinase_pairs"], errors="ignore")

    return df, df_train


In [162]:
phosphomer_pred = pd.read_pickle('/kinase_ml/data/phosformer_pred/interations_e2e_1n1p_db_classed_random_lysate_random.pkl')
phosphomer_pred = phosphomer_pred.loc[:,[ 'substrate_seq', 'kinase_seq', 'p_ind', 'p_aa',
       'kinase_coord', 'label','phosformer_logit']]
df_aug, df_train = make_training_sequences(phosphomer_pred, drop_mismatch=False)
df_train = df_aug.loc[:,['kinase_domain_seq','substrate_seq_15aa','label','phosformer_logit']]
df_train.columns = ['kinase_seq','substrate_seq','label','phosformer_logit']
df_train.to_csv('./evaluation/eval_outputs/phosphormer/interations_e2e_1n1p_db_classed_random_lysate_random.csv')

In [263]:
yeast_pairs = pd.read_pickle('/kinase_ml/data/interactions/humankinase_yeastsub_interactions_cleaned.pkl')
yeast_pairs = yeast_pairs.loc[yeast_pairs['substrate_seq'].apply(lambda s: len(s)==15),:]
yeast_pairs['p_ind'] = 8
yeast_pairs['p_aa'] = 'Y'
yeast_pairs['label'] = True
df_aug, df_train = make_training_sequences(yeast_pairs, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [264]:
pairs_test_neg = pd.concat([pd.read_csv('/ksi_ml/data/splits_v2/test.csv'),
                            pd.read_csv('/ksi_ml/data/splits_v2/test2.csv'),
                           pd.read_csv('/ksi_ml/data/splits_v2/val.csv')])
pairs_test_neg = pairs_test_neg.loc[pairs_test_neg['label']==0,:].drop_duplicates()

In [265]:
pairs_test_neg.loc[ (pairs_test_neg['kinase_seq'] == 'LRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQVMKKLRHEKLVQLYAVVSEEPIYIVTEYMSKGSLLDFLKGETGKYLRLPQLVDMAAQIASGMAYVERMNYVHRDLRAANILVGENLVCKVADFGLARLIEDNEYTARQGAKFPIKWTAPEAALYGRFTIKSDVWSFGILLTELTTKGRVPYPGMVNREVLDQVERGYRMPCPPECPESLHDLMCQCWRKEPEERPTFEYLQAFLEDYF')
    &(pairs_test_neg['substrate_seq']=='LPDSENHYHTFLLWA'),:]


,kinase_seq,substrate_seq,label
1950,LRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQ...,LPDSENHYHTFLLWA,0


In [266]:
pairs_test_neg_aug_v2 = pd.merge(pairs_test_neg,pairs_aug_v2,left_on = ['kinase_seq','substrate_seq'],right_on = ['kinase_domain_seq','substrate_seq_15aa']).drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])


In [267]:
pairs_test_neg_aug_v2.loc[ (pairs_test_neg_aug_v2['kinase_domain_seq'] == 'LRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQVMKKLRHEKLVQLYAVVSEEPIYIVTEYMSKGSLLDFLKGETGKYLRLPQLVDMAAQIASGMAYVERMNYVHRDLRAANILVGENLVCKVADFGLARLIEDNEYTARQGAKFPIKWTAPEAALYGRFTIKSDVWSFGILLTELTTKGRVPYPGMVNREVLDQVERGYRMPCPPECPESLHDLMCQCWRKEPEERPTFEYLQAFLEDYF')
    &(pairs_test_neg_aug_v2['substrate_seq_15aa']=='LPDSENHYHTFLLWA'),:]

,kinase_seq_x,substrate_seq_x,label_x,substrate,substrate_seq_y,kinase,kinase_seq_y,p_ind,p_aa,kinase_coord,label_y,kinase_domain_seq,substrate_seq_15aa,p_aa_match
1104,LRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQ...,LPDSENHYHTFLLWA,0,Q8TCY9,MASPGIEVELLGKGHSDLGEVAPEIKASERRTAVAIADLEWREMEG...,A0A8C0SX18,MGSNKSKPKDASQRRRSLEPAENTHGGGGGAFPTSQTPSKPASADG...,224,Y,"[['287', '540']]",0,LRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQ...,LPDSENHYHTFLLWA,True


In [268]:
yeast_pairs_neg = pd.merge(df_aug.loc[:,['kinase_domain_seq']],pairs_test_neg_aug_v2,
                           left_on = 'kinase_domain_seq',
                               right_on = 'kinase_domain_seq')

In [271]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])

In [272]:
yeast_pairs_neg.columns

Index(['kinase_domain_seq', 'kinase_seq_x', 'substrate_seq_x', 'label_x',
       'substrate', 'substrate_seq_y', 'kinase', 'kinase_seq_y', 'p_ind',
       'p_aa', 'kinase_coord', 'label_y', 'substrate_seq_15aa', 'p_aa_match'],
      dtype='object')

In [273]:
yeast_pairs_neg = yeast_pairs_neg.drop(['kinase_seq_x','substrate_seq_x','label_x'],axis = 1)

In [274]:
yeast_pairs_neg.columns = ['kinase_domain_seq',
       'substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind',
       'p_aa', 'kinase_coord', 'label', 'substrate_seq_15aa', 'p_aa_match']

In [275]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa','kinase_seq'])

In [279]:
yeast_pairs_neg = yeast_pairs_neg.loc[:,['substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind', 'p_aa',
       'kinase_coord', 'label', 'kinase_domain_seq', 'substrate_seq_15aa',
       'p_aa_match']]

In [284]:
yeast_pairs_pos_neg = pd.concat([df_aug,yeast_pairs_neg])

In [285]:
df_aug, df_train = make_training_sequences(yeast_pairs_pos_neg, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [288]:
df_aug.to_csv('/ksi_ml/data/pairs_aug_yeast.csv')
df_train.to_csv('/ksi_ml/data/pairs_yeast.csv')

In [289]:
phosformer_yeast = pd.read_pickle('/kinase_ml/data/phosformer_pred/yeast_human_1n1p.pkl')
phosformer_yeast = phosformer_yeast.loc[:,['substrate_seq', 'kinase_seq', 'p_ind', 'p_aa',
       'kinase_coord', 'label','phosformer_logit','phosformer_pred']]
phosformer_yeast = phosformer_yeast.loc[phosformer_yeast['label']==1,:]
df_phosphomer_pred, _ = make_training_sequences(phosformer_yeast, drop_mismatch=False,
                                                   kinase_domain_ready = True)
df_phosphomer_pred = df_phosphomer_pred.loc[:,['kinase_domain_seq','substrate_seq_15aa','label','phosformer_logit']]
df_phosphomer_pred.to_csv('./evaluation/eval_outputs/phosphormer/phosformer_yeast_orig.csv')

In [9]:
phosformer_train_overlap = pd.read_pickle('/kinase_ml/data/phosformer_pred/kinbert_phosformer_train_overlap.pkl')
df_aug, df_train = make_training_sequences(phosformer_train_overlap, drop_mismatch=True)
df_train = df_aug.loc[:,['kinase_domain_seq','substrate_seq_15aa','label','phosformer_logit']]
df_train.columns = ['kinase_seq','substrate_seq','label','phosformer_logit']
df_train.to_csv('./evaluation/eval_outputs/phosphormer/kinbert_phosformer_train_overlap.csv')
df_aug.to_csv('/ksi_ml/data/pairs_aug_phosphomer_train.csv')
df_train.to_csv('/ksi_ml/data/pairs_phosphomer_train.csv')

In [10]:
phosformer_test_overlap = pd.read_pickle('/kinase_ml/data/phosformer_pred/kinbert_phosformer_test_overlap.pkl')
df_aug, df_train = make_training_sequences(phosformer_test_overlap, drop_mismatch=True)
df_train = df_aug.loc[:,['kinase_domain_seq','substrate_seq_15aa','label','phosformer_logit']]
df_train.columns = ['kinase_seq','substrate_seq','label','phosformer_logit']
df_aug.to_csv('/ksi_ml/data/pairs_aug_phosphomer_test.csv')
df_train.to_csv('./evaluation/eval_outputs/phosphormer/kinbert_phosformer_test_overlap.csv')
df_train.to_csv('/ksi_ml/data/pairs_phosphomer_test.csv')

## Datasets with no psite information and need to be inferred

In [227]:
inter_test = pd.read_pickle('/kinase_ml/data/interations_test_1n1p.pkl')

In [250]:
inter_test_ms = inter_test.loc[inter_test['source']=='ms',:]
df_aug, df_train = make_training_sequences(inter_test_ms, drop_mismatch=True)
df_train.to_csv('/ksi_ml/data/pairs_linkphinder.csv')
df_aug.to_csv('/ksi_ml/data/pairs_aug_linkphinder.csv')

In [251]:
inter_test_y2h = inter_test.loc[inter_test['source']=='y2h',:]
df_aug, df_train = make_training_sequences(inter_test_y2h, drop_mismatch=True)
df_train.to_csv('/ksi_ml/data/pairs_y2h.csv')
df_aug.to_csv('/ksi_ml/data/pairs_aug_y2h.csv')

In [252]:
inter_test_phosphonetwork = inter_test.loc[inter_test['source']=='phosphonetwork',:]
df_aug, df_train = make_training_sequences(inter_test_phosphonetwork, drop_mismatch=True)
df_train.to_csv('/ksi_ml/data/pairs_phosphonetwork.csv')
df_aug.to_csv('/ksi_ml/data/pairs_aug_phosphonetwork.csv')

## precompute all the kinase pdb cache for structbias inference

In [1]:
ls /ksi_ml/data/splits_dark_nondark

test.csv  train.csv  val.csv


In [13]:
dat_name_ls = ['pairs_v1.csv','pairs_v2.csv','pairs_y2h.csv','pairs_yeast.csv','pairs_phosphonetwork.csv','pairs_linkphinder.csv']

In [14]:
dat_ls = []
for dat_name in dat_name_ls:
    dat_path = '/ksi_ml/data/' + dat_name
    dat = pd.read_csv(dat_path).iloc[:,1:]
    dat_ls.append(dat)

In [15]:
dat_all = pd.concat(dat_ls)

In [19]:
dat_all_1perkinase = dat_all.groupby('kinase_seq').head(1)

In [20]:
dat_all_1perkinase.to_csv('/ksi_ml/data/pairs_one_per_kinase_for_eval.csv')

In [ ]:
python fold_kinases_from_pair_csv.py \
  --input /ksi_ml/data/pairs_one_per_kinase_for_eval.csv \
  --out_dir /ksi_ml/data/esmfold_out \
  --batch_size 1 \
  --skip_existing

## Datasets with no kinase information and need to be predict against kinases

In [101]:
import pandas as pd
import numpy as np
from itertools import product

def _parse_kinase_coord(x):
    """
    Accepts things like:
      - [[13, 271]]
      - "[[13, 271]]"
      - "13,271"  (fallback)
    Returns list of (start,end) tuples (1-indexed, inclusive).
    """
    import re
    import ast
    
    if isinstance(x, list):
        coord = x
    elif isinstance(x, str):
        s = x.strip()
        try:
            coord = ast.literal_eval(s)
        except Exception:
            nums = list(map(int, re.findall(r"-?\d+", s)))
            if len(nums) >= 2:
                coord = [[nums[0], nums[1]]]
            else:
                coord = []
    else:
        coord = []

    pairs = []
    for item in coord:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            pairs.append((int(item[0]), int(item[1])))
    return pairs

def _extract_domain(seq: str, coords, join_char: str = ""):
    """
    Extract kinase domain based on coords = list of (start,end) 1-indexed inclusive.
    If multiple segments, concatenates them in order (default join_char="").
    """
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    if not coords:
        return np.nan

    parts = []
    for (start, end) in coords:
        if start < 1 or end < 1:
            continue
        if end < start:
            start, end = end, start
        parts.append(seq[start-1:end])
    if not parts:
        return np.nan
    return join_char.join(parts)

def _extract_15mer_centered(seq: str, p_ind_1: int, p_aa: str, pad_char: str = ""):
    """
    15-aa window centered at p_ind_1 (1-indexed). Inclusive window [p-7, p+7].
    Pads with pad_char if near termini.
    Returns (window, ok_match) where ok_match indicates seq[p_ind_1-1]==p_aa (if possible).
    """
    if not isinstance(seq, str) or len(seq) == 0 or pd.isna(p_ind_1):
        return (np.nan, False)

    p = int(p_ind_1)
    if p < 1:
        return (np.nan, False)

    ok = True
    if isinstance(p_aa, str) and len(p_aa) == 1 and p <= len(seq):
        ok = (seq[p-1] == p_aa)

    left = p - 7
    right = p + 7

    chars = []
    for i in range(left, right + 1):
        if 1 <= i <= len(seq):
            chars.append(seq[i-1])
        else:
            chars.append(pad_char)
    return ("".join(chars), ok)

def create_all_kinase_substrate_pairs(
    kinase_df: pd.DataFrame,
    substrate_df: pd.DataFrame,
    kinase_seq_col: str = "kinase_seq",
    kinase_coord_col: str = "kinase_coord",
    substrate_seq_col: str = "substrate_seq",
    p_ind_col: str = "p_ind",
    p_aa_col: str = "p_aa",
    label_col: str = "label",
    pad_char: str = "",
    drop_mismatch: bool = False,
    kinase_domain_ready: bool = False,
):
    """
    Perform an outer join between all kinases and all substrates to create all possible pairs.
    
    Parameters:
    -----------
    kinase_df : pd.DataFrame
        Dataframe with kinase information (kinase_gene, kinase_name, kinase_seq, kinase_coord, etc.)
    substrate_df : pd.DataFrame
        Dataframe with substrate information (substrate_seq, p_ind, p_aa, label, etc.)
    kinase_seq_col : str
        Column name for kinase sequence
    kinase_coord_col : str
        Column name for kinase domain coordinates
    substrate_seq_col : str
        Column name for substrate sequence
    p_ind_col : str
        Column name for phosphorylation site index
    p_aa_col : str
        Column name for phosphorylation amino acid
    label_col : str
        Column name for labels (if present in substrate_df, otherwise will be NaN)
    pad_char : str
        Character for padding substrate sequences
    drop_mismatch : bool
        Drop rows where p_aa doesn't match sequence
    kinase_domain_ready : bool
        If True, kinase_seq already contains domain sequence (skip extraction)
    
    Returns:
    --------
    df_pairs : pd.DataFrame
        All possible kinase-substrate pairs with processed sequences
    df_train : pd.DataFrame
        Training format with columns: kinase_seq (domain), substrate_seq (15aa), label
    """
    
    # Prepare kinase dataframe
    kinase_prep = kinase_df.copy()
    kinase_prep['_kinase_idx'] = range(len(kinase_prep))
    
    # Prepare substrate dataframe
    substrate_prep = substrate_df.copy()
    substrate_prep['_substrate_idx'] = range(len(substrate_prep))
    
    # Create cross join (cartesian product)
    # Add a temporary key for cross join
    kinase_prep['_merge_key'] = 1
    substrate_prep['_merge_key'] = 1
    
    df_pairs = pd.merge(kinase_prep, substrate_prep, on='_merge_key', suffixes=('_kinase', '_substrate'))
    df_pairs = df_pairs.drop(columns=['_merge_key'])
    
    print(f"Created {len(df_pairs):,} kinase-substrate pairs from {len(kinase_df)} kinases and {len(substrate_df)} substrates")
    
    # Extract kinase domain sequences
    if not kinase_domain_ready:
        df_pairs["_kinase_pairs"] = df_pairs[kinase_coord_col].apply(_parse_kinase_coord)
        df_pairs["kinase_domain_seq"] = [
            _extract_domain(seq, pairs)
            for seq, pairs in zip(df_pairs[kinase_seq_col], df_pairs["_kinase_pairs"])
        ]
        df_pairs = df_pairs.drop(columns=["_kinase_pairs"], errors="ignore")
    else:
        df_pairs["kinase_domain_seq"] = df_pairs[kinase_seq_col]
    
    # Extract substrate 15-mer sequences
    out = [
        _extract_15mer_centered(seq, pind, paa, pad_char=pad_char)
        for seq, pind, paa in zip(
            df_pairs[substrate_seq_col], 
            df_pairs[p_ind_col], 
            df_pairs[p_aa_col]
        )
    ]
    df_pairs["substrate_seq_15aa"] = [x[0] for x in out]
    df_pairs["p_aa_match"] = [x[1] for x in out]
    
    # Optionally drop mismatches
    if drop_mismatch:
        before_drop = len(df_pairs)
        df_pairs = df_pairs[df_pairs["p_aa_match"] == True].copy()
        print(f"Dropped {before_drop - len(df_pairs):,} rows due to p_aa mismatch")
    
    # Create training dataframe
    train_cols = ["kinase_domain_seq", "substrate_seq_15aa"]
    if label_col in df_pairs.columns:
        train_cols.append(label_col)
    
    df_train = df_pairs[train_cols].copy()
    df_train = df_train.rename(
        columns={
            "kinase_domain_seq": "kinase_seq",
            "substrate_seq_15aa": "substrate_seq",
        }
    )
    
    if label_col in df_train.columns:
        df_train = df_train.rename(columns={label_col: "label"})
    else:
        # If no label column, add NaN labels
        df_train["label"] = np.nan
    
    # Drop NAs in sequences
    df_train = df_train.dropna(subset=["kinase_seq", "substrate_seq"]).reset_index(drop=True)
    
    print(f"Final training dataframe: {len(df_train):,} pairs")
    
    return df_pairs, df_train


def create_all_pairs_simple(
    kinase_df: pd.DataFrame,
    substrate_df: pd.DataFrame,
):
    """
    Simple cross join without any sequence processing.
    Just creates all possible kinase-substrate combinations.
    
    Returns dataframe with all columns from both input dataframes.
    """
    kinase_prep = kinase_df.copy()
    substrate_prep = substrate_df.copy()
    
    kinase_prep['_merge_key'] = 1
    substrate_prep['_merge_key'] = 1
    
    df_pairs = pd.merge(kinase_prep, substrate_prep, on='_merge_key', suffixes=('_kinase', '_substrate'))
    df_pairs = df_pairs.drop(columns=['_merge_key'])
    
    print(f"Created {len(df_pairs):,} pairs from {len(kinase_df)} kinases and {len(substrate_df)} substrates")
    
    return df_pairs

In [102]:
ptm_var = pd.read_csv("/kinase_ml/data/labeled_data_full/PTMVar.csv",encoding='latin1',skiprows=6).iloc[:,:26]

In [103]:
ptm_var = ptm_var.loc[ptm_var["MOD_TYPE"] == "Phosphorylation",:]

In [104]:
# ptm_var.to_csv('./PTMVar_phosphorylation.csv')

In [105]:
ptm_var = ptm_var.loc[:,["GENE","UPID","WT_AA","MUT_RSD#","VAR_AA","VAR_TYPE","VAR_CLASS","VAR_POSITION",'dbSNP','FTID']]

In [106]:
ptm_var["VAR_POSITION"] = ptm_var["VAR_POSITION"].apply(lambda s : int(s[1:]))

In [107]:
ptm_var["p_ind"] = ptm_var["MUT_RSD#"]-ptm_var["VAR_POSITION"]

In [108]:
ptm_var.columns = ["substrate_gene","substrate","wt_aa","mut_ind","mut_aa","var_type","var_class","var_dist",'dbSNP','FTID',"p_ind"]

In [109]:
ptm_var.groupby('var_type').count()

,substrate_gene,substrate,wt_aa,mut_ind,mut_aa,var_class,var_dist,dbSNP,FTID,p_ind
var_type,,,,,,,,,,
Disease,2295,2295,2295,2295,2295,2295,2295,2295,2295,2295
LB/B,10790,10790,10790,10790,10790,10790,10790,10790,10790,10790
LP/P,7449,7449,7449,7449,7449,7449,7449,7449,7449,7449
US,3518,3518,3518,3518,3518,3518,3518,3518,3518,3518


In [111]:
# ptm_var = ptm_var.loc[(ptm_var['var_type'].isin(['Disease','LP/P']))&(ptm_var['var_dist'] == 0),:]

In [112]:
ptm_var_subseq = pd.read_csv('./pathogenic_ptmvar_substrate_uniprot_mapping.tsv',sep = '\t').loc[:,['From','Entry Name','Sequence']]

In [113]:
ptm_var = pd.merge(ptm_var,ptm_var_subseq,left_on = 'substrate',right_on = 'From')

In [114]:
ptm_var = ptm_var.drop(['From','Entry Name'],axis = 1)

In [115]:
ptm_var.columns = ['substrate_gene', 'substrate', 'wt_aa', 'mut_ind', 'mut_aa', 'var_type',
       'var_class', 'var_dist','dbSNP', 'FTID', 'p_ind', 'substrate_seq']

In [116]:
ptm_var['label'] = 1

In [121]:
import pandas as pd
import numpy as np

WINDOW = 15
HALF = WINDOW // 2  # 7 for a 15-mer

def make_mut_seq(seq: str, mut_ind: int, wt_aa: str, mut_aa: str) -> str:
    """Return full-length mutant sequence with a single substitution at mut_ind (1-based)."""
    if pd.isna(seq) or not isinstance(seq, str) or pd.isna(mut_ind):
        return np.nan
    i = int(mut_ind) - 1  # convert 1-based -> 0-based
    if i < 0 or i >= len(seq):
        return np.nan

    # Optional sanity check (won't crash your pipeline)
    if isinstance(wt_aa, str) and len(wt_aa) == 1 and seq[i] != wt_aa:
        # You can print/log this if you want:
        # print(f"WT mismatch at {mut_ind}: seq has {seq[i]}, wt_aa is {wt_aa}")
        pass

    return seq[:i] + str(mut_aa) + seq[i+1:]


def extract_15mer(seq: str, center_pos: int, window: int = 15) -> str:
    """Extract window-length substring centered at center_pos (1-based). Returns shorter if near ends."""
    if pd.isna(seq) or not isinstance(seq, str) or pd.isna(center_pos):
        return np.nan
    center = int(center_pos) - 1  # 0-based
    if center < 0 or center >= len(seq):
        return np.nan

    half = window // 2
    start = max(0, center - half)
    end = min(len(seq), center + half + 1)  # +1 because Python slicing end is exclusive
    return seq[start:end]

In [125]:
# 1) Build full-length mutant sequence
df = ptm_var
df["substrate_seq_mut"] = ptm_var.apply(
    lambda r: make_mut_seq(r["substrate_seq"], r["mut_ind"], r["wt_aa"], r["mut_aa"]),
    axis=1
)

# 2) Extract 15aa window centered at p_ind from WT and MUT
df["substrate_15aa_wt"] = df.apply(
    lambda r: extract_15mer(r["substrate_seq"], r["p_ind"], WINDOW),
    axis=1
)
df["substrate_15aa_mut"] = df.apply(
    lambda r: extract_15mer(r["substrate_seq_mut"], r["p_ind"], WINDOW),
    axis=1
)

In [129]:
df['substrate_15aa_wt'] = df['substrate_15aa_wt'].astype('str')

In [134]:
df = df.loc[df['substrate_15aa_wt'].apply(lambda s: len(s)==15),:]

In [137]:
df['p_aa'] = df['substrate_15aa_wt'].apply(lambda s: s[7])

/tmp/ipykernel_24213/2752853625.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['p_aa'] = df['substrate_15aa_wt'].apply(lambda s: s[7])


In [141]:
df = df.loc[df['p_aa'].isin(['S','T','Y']),:]

In [142]:
ptm_var = df.copy()

In [143]:
ptm_var_psite = ptm_var.loc[ptm_var['wt_aa'].isin(['S','T','Y'])&(ptm_var['var_dist']==0),:]

In [146]:
ptm_var_psite = ptm_var_psite.loc[ptm_var_psite['var_class']=='I',:]

In [147]:
ptm_var_flank = ptm_var.loc[(ptm_var['var_dist']!=0),:]

In [148]:
ptm_var_flank

,substrate_gene,substrate,wt_aa,mut_ind,mut_aa,var_type,var_class,var_dist,dbSNP,FTID,p_ind,substrate_seq,label,substrate_seq_mut,substrate_15aa_wt,substrate_15aa_mut,p_aa
0,A1CF,Q9NQ94,V,555,M,LB/B,II,3,rs9073,VAR_052201,552,MESNHKSGDGLSGTQKEAALRALVQRTGYSLVQENGQRKYGGPPPG...,1,MESNHKSGDGLSGTQKEAALRALVQRTGYSLVQENGQRKYGGPPPG...,GYAVPNATAPVSAAQ,GYAVPNATAPMSAAQ,T
1,A2M,P01023,L,815,Q,LB/B,II,-2,rs3180392,VAR_026821,817,MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...,1,MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...,LKATVLNYLPKCIRV,LKATVQNYLPKCIRV,Y
2,A2M,P01023,R,704,H,LB/B,II,-4,rs1800434,VAR_000012,708,MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...,1,MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...,EGLRVGFYESDVMGR,EGLHVGFYESDVMGR,Y
4,A4GALT,Q9NPC4,V,316,M,Disease,II,-5,-,Br08X,321,MSKPPDLLLRLLRGAPRQRVCTLFIIGFKFTFFVSIMIYWHVVGEP...,1,MSKPPDLLLRLLRGAPRQRVCTLFIIGFKFTFFVSIMIYWHVVGEP...,VHVWNKKSQGTRFEA,VHMWNKKSQGTRFEA,S
6,AAAS,Q9NRG9,S,263,P,LP/P,II,2,rs121918550,VAR_012806,261,MCSLGLFPPPPPRGQVTLYEHNNELVTGSSYESPPPDFRGQWINLP...,1,MCSLGLFPPPPPRGQVTLYEHNNELVTGSSYESPPPDFRGQWINLP...,PSGGRLLSASPVDAA,PSGGRLLSAPPVDAA,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24046,ZSCAN5B,A6NJL1,V,208,I,LB/B,II,3,rs4801296,VAR_042966,205,MAANWTLSWGQGGPCNSPGSDTPRSVASPETQLGNHDRNPETWHMN...,1,MAANWTLSWGQGGPCNSPGSDTPRSVASPETQLGNHDRNPETWHMN...,EDFLLHKSIDVTGDP,EDFLLHKSIDITGDP,S
24047,ZSWIM4,Q9H7M6,P,89,L,Disease,II,2,-,Br27P,87,MEPPAAKRSRGCPAGPEERDAGAGAARGRGRPEALLDLSAKRVAES...,1,MEPPAAKRSRGCPAGPEERDAGAGAARGRGRPEALLDLSAKRVAES...,CMYSSLGYPPPEGEH,CMYSSLGYPLPEGEH,Y
24049,ZZEF1,O43149,P,2051,S,LB/B,II,2,rs1006954,VAR_032556,2049,MGNAPSHSSEDEAAAAGGEGWGPHQDWAAVSGTTPGPGVAAPALPP...,1,MGNAPSHSSEDEAAAAGGEGWGPHQDWAAVSGTTPGPGVAAPALPP...,SDSPADASPPTGLPD,SDSPADASPSTGLPD,S
24050,ZZEF1,O43149,E,2369,Q,LB/B,II,-5,rs711177,VAR_032559,2374,MGNAPSHSSEDEAAAAGGEGWGPHQDWAAVSGTTPGPGVAAPALPP...,1,MGNAPSHSSEDEAAAAGGEGWGPHQDWAAVSGTTPGPGVAAPALPP...,GFEEIRATFLQTDLL,GFQEIRATFLQTDLL,T


In [149]:
kinome_dat = pd.read_pickle('/kinase_ml/data/human_kinome_seq_230301.pkl')

In [150]:
df_pairs, df_train = create_all_kinase_substrate_pairs(
        kinase_df=kinome_dat,
        substrate_df=ptm_var_psite,
        kinase_seq_col='kinase_seq',
        kinase_coord_col='kinase_coord',
        substrate_seq_col='substrate_seq',
        p_ind_col='p_ind',
        p_aa_col='p_aa',
        label_col='label',
        pad_char='',
        drop_mismatch=False,
        kinase_domain_ready=False
    )

Created 843,028 kinase-substrate pairs from 503 kinases and 1676 substrates
Final training dataframe: 843,028 pairs


In [151]:
df_train.to_csv('/ksi_ml/data/pairs_ptmvar.csv')
df_pairs.to_csv('/ksi_ml/data/pairs_aug_ptmvar.csv')

In [169]:
kinome_dat['kinase_domain_seq'] = kinome_dat.apply(lambda s: s['kinase_seq'][(int(s['kinase_coord'][0][0])-1):int(s['kinase_coord'][0][1])],axis = 1)
df_pairs = create_all_pairs_simple(kinase_df=kinome_dat,substrate_df=ptm_var_flank)

Created 10,781,805 pairs from 503 kinases and 21435 substrates


In [172]:
df_pairs.to_csv('/ksi_ml/data/pairs_aug_ptmvar_flank.csv')

In [174]:
df_out = df_pairs.loc[:,['kinase_domain_seq','substrate_15aa_wt','label']]
df_out.columns = ['kinase_seq','substrate_seq','label']
df_out.to_csv('/ksi_ml/data/pairs_ptmvar_flank_wt.csv')

In [175]:
df_out = df_pairs.loc[:,['kinase_domain_seq','substrate_15aa_mut','label']]
df_out.columns = ['kinase_seq','substrate_seq','label']
df_out.to_csv('/ksi_ml/data/pairs_ptmvar_flank_mut.csv')

In [181]:
ptm_var_psite.loc[ptm_var_psite['substrate_gene']=='CTNNB1',:]

,substrate_gene,substrate,wt_aa,mut_ind,mut_aa,var_type,var_class,var_dist,dbSNP,FTID,p_ind,substrate_seq,label,substrate_seq_mut,substrate_15aa_wt,substrate_15aa_mut,p_aa
4827,CTNNB1,P35222,S,23,R,US,I,0,rs1413975856,VAR_017612,23,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVRHWQQQSYLDSGIHSGATTTAPSL...,PDRKAAVSHWQQQSY,PDRKAAVRHWQQQSY,S
4841,CTNNB1,P35222,S,45,F,US,I,0,rs121913409,VAR_017631,45,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPFL...,GATTTAPSLSGKGNP,GATTTAPFLSGKGNP,S
4842,CTNNB1,P35222,S,45,P,US,I,0,rs121913407,VAR_017632,45,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPPL...,GATTTAPSLSGKGNP,GATTTAPPLSGKGNP,S
4847,CTNNB1,P35222,T,41,A,US,I,0,rs121913412,VAR_017629; HX13T[COSMIC],41,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATATAPSL...,GIHSGATTTAPSLSG,GIHSGATATAPSLSG,T
4848,CTNNB1,P35222,T,41,I,US,I,0,rs121913413,VAR_017630,41,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATITAPSL...,GIHSGATTTAPSLSG,GIHSGATITAPSLSG,T
4854,CTNNB1,P35222,S,33,F,US,I,0,rs121913400,VAR_017617,33,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDFGIHSGATTTAPSL...,QQQSYLDSGIHSGAT,QQQSYLDFGIHSGAT,S
4855,CTNNB1,P35222,S,33,L,US,I,0,-,VAR_017618,33,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDLGIHSGATTTAPSL...,QQQSYLDSGIHSGAT,QQQSYLDLGIHSGAT,S
4875,CTNNB1,P35222,S,37,A,US,I,0,rs121913228,VAR_017624,37,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHAGATTTAPSL...,YLDSGIHSGATTTAP,YLDSGIHAGATTTAP,S
4876,CTNNB1,P35222,S,37,C,LP/P,I,0,rs121913403,VAR_017625,37,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHCGATTTAPSL...,YLDSGIHSGATTTAP,YLDSGIHCGATTTAP,S
4877,CTNNB1,P35222,S,37,F,LP/P,I,0,rs121913403,VAR_017626; 16979; HX12T[cBio; COSMIC],37,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHSGATTTAPSL...,1,MATQADLMELDMAMEPDRKAAVSHWQQQSYLDSGIHFGATTTAPSL...,YLDSGIHSGATTTAP,YLDSGIHFGATTTAP,S


## Ingest Virus dms and phosphopreoteomics data

In [5]:
virus_substrate = pd.read_pickle("/kinase_ml/data/labeled_data_full/virus_psites/covid_psite_1_min.pkl")

In [14]:
virus_substrate.head(1)

,substrate,p_ind,p_aa,substrate_seq,label
0,NCAP_SARS2,180,S,MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLP...,1


In [6]:
virus_substrate['label'] = 1

In [9]:
df_pairs, df_train = create_all_kinase_substrate_pairs(
        kinase_df=kinome_dat,
        substrate_df=virus_substrate,
        kinase_seq_col='kinase_seq',
        kinase_coord_col='kinase_coord',
        substrate_seq_col='substrate_seq',
        p_ind_col='p_ind',
        p_aa_col='p_aa',
        label_col='label',
        pad_char='',
        drop_mismatch=False,
        kinase_domain_ready=False
    )

Created 22,132 kinase-substrate pairs from 503 kinases and 44 substrates
Final training dataframe: 22,132 pairs


In [126]:
df_train.to_csv('/ksi_ml/data/pairs_virus.csv')
df_pairs.to_csv('/ksi_ml/data/pairs_aug_virus.csv')

In [17]:
import pandas as pd
virus_substrate_dms = pd.read_pickle("/kinase_ml/data/virus_dms/dms_candidate_psites.pkl")

In [18]:
virus_substrate_dms = virus_substrate_dms.loc[:,['substrate','p_ind','wt_aa','substrate_seq']]
virus_substrate_dms.columns = ['substrate','p_ind','p_aa','substrate_seq']
virus_substrate_dms['label'] = 1

In [20]:
df_pairs, df_train = create_all_kinase_substrate_pairs(
        kinase_df=kinome_dat,
        substrate_df=virus_substrate_dms,
        kinase_seq_col='kinase_seq',
        kinase_coord_col='kinase_coord',
        substrate_seq_col='substrate_seq',
        p_ind_col='p_ind',
        p_aa_col='p_aa',
        label_col='label',
        pad_char='',
        drop_mismatch=False,
        kinase_domain_ready=False
    )

Created 60,360 kinase-substrate pairs from 503 kinases and 120 substrates
Final training dataframe: 60,360 pairs


In [21]:
df_train.to_csv('/ksi_ml/data/pairs_virus_dms.csv')
df_pairs.to_csv('/ksi_ml/data/pairs_aug_virus_dms.csv')